In [1]:
import numpy as np
from scipy.integrate import solve_ivp

# Physical Constants (CGS)
G = 6.67430e-8      # cm^3 g^-1 s^-2
c = 2.99792e10      # cm s^-1
M_sun = 1.989e33    # g
km = 1e5            # cm

def pulsar_properties(P, P_dot):
    """
    Calculates surface dipole magnetic field and characteristic spin-down age.
    Inputs:
        P: Spin period (seconds)
        P_dot: Spin-down rate (s/s)
    """
    B = 3.2e19 * np.sqrt(P * P_dot)
    tau_yr = (P / (2.0 * P_dot)) / (365.25 * 86400)
    return B, tau_yr

def kerr_isco_efficiency(a_star):
    """
    Calculates Bardeen-Press-Teukolsky ISCO radius (in GM/c^2) and 
    thin-disc accretion radiative efficiency eta for a Kerr black hole.
    Input:
        a_star: Dimensionless spin parameter cJ/GM^2 (-1 <= a_star <= 1)
    """
    Z1 = 1 + (1 - a_star**2)**(1/3) * ((1 + a_star)**(1/3) + (1 - a_star)**(1/3))
    Z2 = np.sqrt(3 * a_star**2 + Z1**2)
    r_isco = 3 + Z2 - np.sqrt((3 - Z1) * (3 + Z1 + 2 * Z2))
    
    # Binding energy at ISCO -> Radiative efficiency eta = 1 - E_isco
    E_isco = (r_isco**(1.5) - 2*r_isco**(0.5) + a_star) / (
        r_isco**(0.75) * np.sqrt(r_isco**(1.5) - 3*r_isco**(0.5) + 2*a_star)
    )
    eta = 1.0 - E_isco
    return r_isco, eta

def tov_system(r, y, K, gamma):
    """
    Tolman-Oppenheimer-Volkoff (TOV) hydrostatic equilibrium differential equations.
    y[0] = Pressure P
    y[1] = Enclosed mass m
    """
    P, m = y
    if P <= 0:
        return [0, 0]
    
    rho = (P / K)**(1.0 / gamma)
    
    # Relativistic TOV terms
    num = -G * (rho + P / c**2) * (m + 4 * np.pi * r**3 * P / c**2)
    den = r**2 * (1 - 2 * G * m / (r * c**2))
    
    dP_dr = num / den
    dm_dr = 4 * np.pi * r**2 * rho
    return [dP_dr, dm_dr]

def solve_tov_star(rho_c, K=100000.0, gamma=2.0):
    """
    Integrates the TOV system from central density rho_c to the stellar surface.
    """
    P_c = K * rho_c**gamma
    y0 = [P_c, 1e-10]
    
    # Event function to stop integration when Pressure reaches zero (surface)
    def surface_event(r, y, K, gamma):
        return y[0]
    surface_event.terminal = True
    surface_event.direction = -1

    sol = solve_ivp(tov_system, (1e2, 3e6), y0, args=(K, gamma), 
                    events=surface_event, rtol=1e-8, atol=1e-10)
    
    R_km = sol.t[-1] / km
    M_solar = sol.y[1][-1] / M_sun
    return R_km, M_solar

if __name__ == "__main__":
    print("--- COMPACT OBJECT NUMERICAL SUITE ---\n")
    
    # 1. Pulsar Spin-down Analysis (Question 2)
    P_val, P_dot_val = 0.1, 1e-15
    B_field, age = pulsar_properties(P_val, P_dot_val)
    print(f"[Pulsar Analysis]")
    print(f"  P = {P_val} s, P_dot = {P_dot_val} s/s")
    print(f"  -> Surface Magnetic Field B: {B_field:.2e} Gauss")
    print(f"  -> Characteristic Age tau : {age:.2e} years\n")
    
    # 2. ISCO & Accretion Efficiency (Part IV)
    r_isco_0, eta_0 = kerr_isco_efficiency(0.0)
    r_isco_max, eta_max = kerr_isco_efficiency(0.998)
    print(f"[Kerr Black Hole ISCO & Efficiency]")
    print(f"  Non-spinning (a* = 0.000): ISCO = {r_isco_0:.2f} GM/c^2 | Efficiency = {eta_0*100:.2f}%")
    print(f"  Thorne Limit (a* = 0.998): ISCO = {r_isco_max:.2f} GM/c^2 | Efficiency = {eta_max*100:.2f}%\n")
    
    # 3. TOV Neutron Star Structure Model (Part II)
    rho_central = 1.0e15  # g/cm^3
    R_ns, M_ns = solve_tov_star(rho_c=rho_central)
    print(f"[TOV Neutron Star Hydrostatic Model]")
    print(f"  Central Density rho_c : {rho_central:.2e} g/cm^3")
    print(f"  -> Calculated Radius  : {R_ns:.2f} km")
    print(f"  -> Calculated Mass    : {M_ns:.2f} M_sun")

--- COMPACT OBJECT NUMERICAL SUITE ---

[Pulsar Analysis]
  P = 0.1 s, P_dot = 1e-15 s/s
  -> Surface Magnetic Field B: 3.20e+11 Gauss
  -> Characteristic Age tau : 1.58e+06 years

[Kerr Black Hole ISCO & Efficiency]
  Non-spinning (a* = 0.000): ISCO = 6.00 GM/c^2 | Efficiency = 5.72%
  Thorne Limit (a* = 0.998): ISCO = 1.24 GM/c^2 | Efficiency = 32.10%

[TOV Neutron Star Hydrostatic Model]
  Central Density rho_c : 1.00e+15 g/cm^3
  -> Calculated Radius  : 12.48 km
  -> Calculated Mass    : 1.22 M_sun
